# 🔐 Decryption / Encoding Fix Notebook
## File: `liste_des_compétences.xlsx`

### Context
This notebook detects and repairs **encoding corruption** in the dataset.
The 'encrypted' values are text strings that were encoded in **UTF-8** but incorrectly read as **Latin-1 / Windows-1252**, producing garbled characters such as:
- `sÃ©curitÃ©` → `sécurité`
- `RÃ©aliser` → `Réaliser`
- `contrÃ´le` → `contrôle`

The keyword **`crypto`** in this context refers to this **cryptographic-style encoding corruption** pattern.

### Workflow
1. Import Libraries  
2. Load Data  
3. Data Inspection  
4. Decryption Logic (Encoding Fix)  
5. Apply Transformation  
6. Before / After Comparison  
7. Export Results  

---
## 1. 📦 Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

pandas  version : 2.2.3
numpy   version : 2.1.3


---
## 2. 📂 Load Data

In [5]:
# ── Configuration ────────────────────────────────────────────────────────────
INPUT_FILE  = "liste des compétences.xlsx"   # source file
OUTPUT_FILE = "liste_des_competences_clean.csv"  # decrypted output
CRYPTO_KEYWORD = "crypto"  # marker keyword used to identify encrypted columns
                            # (here: we scan for the Ã pattern produced by UTF-8/Latin-1 mismatch)

# ── Load ─────────────────────────────────────────────────────────────────────
assert Path(INPUT_FILE).exists(), f"File not found: {INPUT_FILE}"

df_raw = pd.read_excel(INPUT_FILE, dtype=str)   # read everything as string to preserve raw bytes

print(f"✅ Loaded  {INPUT_FILE}")
print(f"   Shape  : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"   Columns: {df_raw.columns.tolist()}")

✅ Loaded  liste des compétences.xlsx
   Shape  : 158 rows × 12 columns
   Columns: ['Domaine', 'Metier', 'Competence', 'Type_Competence', 'Indicateurs_Observables', 'Modalites_Evaluation', 'Preuves_Attendues', 'Situations_Professionnelles_Type', 'Outils_SI_ou_Materiels', 'Reglementation_ou_Normes', 'Formation_ou_Activites_Pedago', 'Mots_Cles']


---
## 3. 🔍 Data Inspection

In [6]:
# ── General overview ─────────────────────────────────────────────────────────
print("=" * 60)
print("RAW DATA — first 5 rows")
print("=" * 60)
df_raw.head()

RAW DATA — first 5 rows


,Domaine,Metier,Competence,Type_Competence,Indicateurs_Observables,Modalites_Evaluation,Preuves_Attendues,Situations_Professionnelles_Type,Outils_SI_ou_Materiels,Reglementation_ou_Normes,Formation_ou_Activites_Pedago,Mots_Cles
0,1. CONDUIRE,Conducteur Routier (PL/SPL),Appliquer les principes de conduite rationnell...,Technique,"Anticipation, vitesse adaptÃ©e, freinage limit...",Mise en situation (simulateur/terrain) + grill...,Grille observation signÃ©e + relevÃ©s conso/tÃ...,Trajet avec contraintes dÃ©lai + conso,"TÃ©lÃ©matique, ordinateur de bord",SÃ©curitÃ© routiÃ¨re,TP simulateur + analyse donnÃ©es,"eco-conduite, sÃ©curitÃ©, consommation"
1,1. CONDUIRE,Conducteur Routier (PL/SPL),RÃ©aliser des contrÃ´les de base et identifier...,Technique,"Check pneus/niveaux/fuites, signalement cohÃ©r...",QCM + Ã©tude de cas panne + oral,Check-list contrÃ´le + fiche anomalie,Prise de poste et contrÃ´le dÃ©part,"VÃ©hicule, check-list",ProcÃ©dures entreprise,TP contrÃ´le dÃ©part,"maintenance 1er niveau, sÃ©curitÃ©"
2,1. CONDUIRE,Conducteur Routier (PL/SPL),SÃ©curiser le chargement par un arrimage adapt...,Technique,"Sangles/points dâancrage adaptÃ©s, calage, c...",Mise en situation atelier + audit,Check-list arrimage + photos,Chargement palettes hÃ©tÃ©rogÃ¨nes,"Sangles, barres, tapis",Bonnes pratiques arrimage,TP arrimage + RETEX,"arrimage, chargement"
3,1. CONDUIRE,Conducteur Routier (PL/SPL),ComplÃ©ter correctement les documents de trans...,Organisationnelle,"Champs complets, cohÃ©rence dates/poids/ADR, t...",Ãtude de cas + exercice e-CMR,CMR/e-CMR complÃ©tÃ©e + correction,EnlÃ¨vement/livraison multi-clients,"Appli e-CMR, scanner",Convention CMR,TD documentation transport,"CMR, eCMR, traÃ§abilitÃ©"
4,1. CONDUIRE,Conducteur Routier (PL/SPL),GÃ©rer les temps de conduite et repos en confo...,Organisationnelle,"Planning sans infraction, justification alÃ©as...",Ãtude de cas + QCM rÃ©glementaire,Planning + score QCM + relevÃ© tachy simulÃ©,TournÃ©e longue distance,Chronotachygraphe,RSE,TD rÃ©glementation + simulation tournÃ©e,"RSE, tachy, conformitÃ©"


In [7]:
# ── Missing values ───────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_report = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).query("`Missing Count` > 0")

if missing_report.empty:
    print("✅ No missing values found.")
else:
    print("⚠️  Columns with missing values:")
    display(missing_report)

✅ No missing values found.


In [8]:
# ── Detect corrupted / 'crypto-encoded' cells ────────────────────────────────
# The encoding corruption pattern: UTF-8 multibyte chars read as Latin-1
# always produce the sequence  Ã + a second character (e.g. Ã©, Ã¨, Ã´, Ã¢ …)
# This regex reliably identifies those garbled cells.
CORRUPT_PATTERN = re.compile(r'Ã[\x80-\xBF\xa0-\xff©¨´¢¹¼½¾²³°àáâãäåæçèéêëìíîï]', re.UNICODE)

def is_corrupted(value: object) -> bool:
    """Return True if the cell value looks like an encoding-corrupted string."""
    if pd.isna(value):
        return False
    return bool(CORRUPT_PATTERN.search(str(value)))

# Build a boolean mask — True where the cell is corrupted
corrupt_mask = df_raw.apply(lambda col: col.map(is_corrupted))

total_corrupted = corrupt_mask.values.sum()
affected_cols   = corrupt_mask.any().sum()

print(f"🔐 Corrupted ('crypto') cells detected : {total_corrupted}")
print(f"   Across {affected_cols} column(s) out of {df_raw.shape[1]}")
print()
print("Per-column breakdown:")
col_counts = corrupt_mask.sum().rename("Corrupted Cells")
display(col_counts[col_counts > 0].to_frame())

🔐 Corrupted ('crypto') cells detected : 747
   Across 11 column(s) out of 12

Per-column breakdown:


,Corrupted Cells
Domaine,25
Metier,53
Competence,141
Indicateurs_Observables,134
Modalites_Evaluation,77
Preuves_Attendues,49
Situations_Professionnelles_Type,83
Outils_SI_ou_Materiels,20
Reglementation_ou_Normes,71
Formation_ou_Activites_Pedago,23


---
## 4. 🔓 Decryption Logic

The 'decryption' here is a **re-encoding** operation:
- The text was stored as **UTF-8** bytes
- But was mistakenly read as **Latin-1 / Windows-1252**
- Fix: re-encode back to Latin-1 bytes → decode as UTF-8

In [14]:
import re

# Pattern to detect encoding corruption (both Ã and â families)
CORRUPT_PATTERN = re.compile(r'[ÃÂ][^\s]', re.UNICODE)

# Pattern to catch leftover â artifacts after decryption
ARTIFACT_PATTERN = re.compile(r'â\S*')

def is_corrupted(value: object) -> bool:
    """Return True if the cell value looks like an encoding-corrupted string."""
    if pd.isna(value):
        return False
    return bool(CORRUPT_PATTERN.search(str(value)))


def decrypt_cell(value: object, fallback: str = "latin-1", target: str = "utf-8") -> object:
    """
    Repair a single cell that suffered a UTF-8 → Latin-1 mis-decoding.

    Parameters
    ----------
    value    : raw cell value (any type).
    fallback : encoding that was *wrongly* applied  (default 'latin-1').
    target   : encoding that was *originally* used  (default 'utf-8').

    Returns
    -------
    Corrected string, or the original value if repair fails or is not needed.
    """
    if pd.isna(value):
        return value                  # preserve NaN / None as-is
    if not isinstance(value, str):
        return value                  # non-string cells are left untouched
    if not is_corrupted(value):
        return value                  # clean cells are left untouched
    try:
        # Step 1: re-encode the string using the wrong encoding → get the raw bytes back
        # Step 2: decode those bytes with the correct encoding
        repaired = value.encode(fallback).decode(target)

        # Step 3: remove any leftover â artifacts that survive the re-encoding
        repaired = ARTIFACT_PATTERN.sub("", repaired)

        # Step 4: clean up any double spaces left behind after removal
        repaired = re.sub(r' {2,}', ' ', repaired).strip()

        return repaired
    except (UnicodeDecodeError, UnicodeEncodeError):
        # If the round-trip fails, return the original value unchanged
        return value


def decrypt_dataframe(
    df: pd.DataFrame,
    fallback: str = "latin-1",
    target: str = "utf-8"
) -> pd.DataFrame:
    """
    Apply decrypt_cell to every cell of a DataFrame using vectorized applymap.
    Non-string and clean cells are left untouched.

    Parameters
    ----------
    df       : input DataFrame (values read as raw strings).
    fallback : the encoding that was incorrectly applied.
    target   : the original correct encoding.

    Returns
    -------
    A new DataFrame with all corrupted cells repaired.
    """
    return df.apply(
        lambda col: col.map(lambda v: decrypt_cell(v, fallback, target))
    )


print("✅ Decryption functions defined.")

# Quick sanity-check on sample values
samples = [
    "Appliquer les principes de conduite rationnelle pour rÃ©duire consommation et risques",
    "eco-conduite, sÃ©curitÃ©, consommation",
    "RÃ©aliser des contrÃ´les de base et identifier des anomalies mÃ©caniques simples",
    "Gestion câurgence et soins",           # â artifact example
    "Normal text with no issue",            # should remain unchanged
    None,                                   # NaN → should remain None
]

print("\nSanity-check on hard-coded samples:")
print(f"{'BEFORE':<75}  →  AFTER")
print("-" * 110)
for s in samples:
    result = decrypt_cell(s)
    print(f"{str(s):<75}  →  {str(result)}")

✅ Decryption functions defined.

Sanity-check on hard-coded samples:
BEFORE                                                                       →  AFTER
--------------------------------------------------------------------------------------------------------------
Appliquer les principes de conduite rationnelle pour rÃ©duire consommation et risques  →  Appliquer les principes de conduite rationnelle pour réduire consommation et risques
eco-conduite, sÃ©curitÃ©, consommation                                       →  eco-conduite, sécurité, consommation
RÃ©aliser des contrÃ´les de base et identifier des anomalies mÃ©caniques simples  →  Réaliser des contrôles de base et identifier des anomalies mécaniques simples
Gestion câurgence et soins                                                   →  Gestion câurgence et soins
Normal text with no issue                                                    →  Normal text with no issue
None                                                              

---
## 5. ⚙️ Apply Transformation

In [15]:
# ── Run decryption across the full DataFrame ─────────────────────────────────
df_clean = decrypt_dataframe(df_raw)

# ── Verify no corrupted cells remain ─────────────────────────────────────────
still_corrupt = df_clean.apply(lambda col: col.map(is_corrupted)).values.sum()

if still_corrupt == 0:
    print(f"✅ Decryption complete — 0 corrupted cells remaining.")
else:
    print(f"⚠️  {still_corrupt} cell(s) could not be fully repaired (see inspection above).")

print(f"   Processed: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

✅ Decryption complete — 0 corrupted cells remaining.
   Processed: 158 rows × 12 columns


---
## 6. 🔄 Before / After Comparison

In [16]:
# ── Show a side-by-side diff for every changed cell ──────────────────────────
changed_cells = []
for col in df_raw.columns:
    before_series = df_raw[col].astype(str)
    after_series  = df_clean[col].astype(str)
    diff_idx = before_series[before_series != after_series].index
    for idx in diff_idx:
        changed_cells.append({
            "Row": idx,
            "Column": col,
            "BEFORE (corrupted)": df_raw.at[idx, col],
            "AFTER  (decrypted)": df_clean.at[idx, col],
        })

diff_df = pd.DataFrame(changed_cells)

print(f"Total cells modified: {len(diff_df)}")
print()

# Display a sample of up to 20 changes
sample_diff = diff_df.head(20)
pd.set_option("display.max_colwidth", 90)
display(sample_diff)

Total cells modified: 745



,Row,Column,BEFORE (corrupted),AFTER (decrypted)
0,60,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
1,61,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
2,62,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
3,63,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
4,64,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
5,65,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
6,66,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
7,67,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
8,68,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR
9,69,Domaine,3. RÃPARER & ENTRETENIR,3. RÉPARER & ENTRETENIR


In [18]:
# ── Full cleaned preview ──────────────────────────────────────────────────────
print("=" * 60)
print("CLEANED DATA — first 5 rows")
print("=" * 60)
df_clean.head()

CLEANED DATA — first 5 rows


,Domaine,Metier,Competence,Type_Competence,Indicateurs_Observables,Modalites_Evaluation,Preuves_Attendues,Situations_Professionnelles_Type,Outils_SI_ou_Materiels,Reglementation_ou_Normes,Formation_ou_Activites_Pedago,Mots_Cles
0,1. CONDUIRE,Conducteur Routier (PL/SPL),Appliquer les principes de conduite rationnelle pour réduire consommation et risques,Technique,"Anticipation, vitesse adaptée, freinage limité, baisse conso mesurable",Mise en situation (simulateur/terrain) + grille observation,Grille observation signée + relevés conso/télématique,Trajet avec contraintes délai + conso,"Télématique, ordinateur de bord",Sécurité routière,TP simulateur + analyse données,"eco-conduite, sécurité, consommation"
1,1. CONDUIRE,Conducteur Routier (PL/SPL),Réaliser des contrôles de base et identifier des anomalies mécaniques simples,Technique,"Check pneus/niveaux/fuites, signalement cohérent, arrêt si danger",QCM + étude de cas panne + oral,Check-list contrôle + fiche anomalie,Prise de poste et contrôle départ,"Véhicule, check-list",Procédures entreprise,TP contrôle départ,"maintenance 1er niveau, sécurité"
2,1. CONDUIRE,Conducteur Routier (PL/SPL),Sécuriser le chargement par un arrimage adapté au type de marchandise,Technique,"Sangles/points d’ancrage adaptés, calage, contrôle tension",Mise en situation atelier + audit,Check-list arrimage + photos,Chargement palettes hétérogènes,"Sangles, barres, tapis",Bonnes pratiques arrimage,TP arrimage + RETEX,"arrimage, chargement"
3,1. CONDUIRE,Conducteur Routier (PL/SPL),Compléter correctement les documents de transport (CMR/e-CMR) et traçabilité,Organisationnelle,"Champs complets, cohérence dates/poids/ADR, traçabilité",Étude de cas + exercice e-CMR,CMR/e-CMR complétée + correction,Enlèvement/livraison multi-clients,"Appli e-CMR, scanner",Convention CMR,TD documentation transport,"CMR, eCMR, traçabilité"
4,1. CONDUIRE,Conducteur Routier (PL/SPL),Gérer les temps de conduite et repos en conformité RSE,Organisationnelle,"Planning sans infraction, justification aléas, usage tachy correct",Étude de cas + QCM réglementaire,Planning + score QCM + relevé tachy simulé,Tournée longue distance,Chronotachygraphe,RSE,TD réglementation + simulation tournée,"RSE, tachy, conformité"


---
## 7. 💾 Export Results

In [21]:
# ── Export to CSV with proper UTF-8 encoding ─────────────────────────────────
df_clean.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",   # utf-8-sig adds BOM → opens correctly in Excel
    sep=","
)

output_size = Path(OUTPUT_FILE).stat().st_size / 1024
print(f"✅ Exported → {OUTPUT_FILE}")
print(f"   Rows    : {len(df_clean)}")
print(f"   Columns : {len(df_clean.columns)}")
print(f"   Size    : {output_size:.1f} KB")
print(f"   Encoding: UTF-8 with BOM (excel-safe)")

✅ Exported → liste_des_competences_clean.csv
   Rows    : 158
   Columns : 12
   Size    : 44.7 KB
   Encoding: UTF-8 with BOM (excel-safe)


In [20]:
# ── Final summary report ──────────────────────────────────────────────────────
print("=" * 60)
print("SUMMARY REPORT")
print("=" * 60)
print(f"  Input file        : {INPUT_FILE}")
print(f"  Output file       : {OUTPUT_FILE}")
print(f"  Dataset shape     : {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
print(f"  Cells inspected   : {df_clean.size}")
print(f"  Cells decrypted   : {len(diff_df)}")
print(f"  Cells unchanged   : {df_clean.size - len(diff_df)}")
print(f"  Remaining corrupt : {still_corrupt}")
print("=" * 60)

SUMMARY REPORT
  Input file        : liste des compétences.xlsx
  Output file       : liste_des_competences_clean.csv
  Dataset shape     : 158 rows × 12 columns
  Cells inspected   : 1896
  Cells decrypted   : 745
  Cells unchanged   : 1151
  Remaining corrupt : 0
